In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('Dataset.csv')
df.head(10)

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"
5,s9,TV Show,The Great British Baking Show,Andy Devonshire,United Kingdom,9/24/2021,2021,TV-14,9 Seasons,"British TV Shows, Reality TV"
6,s10,Movie,The Starling,Theodore Melfi,United States,9/24/2021,2021,PG-13,104 min,"Comedies, Dramas"
7,s939,Movie,Motu Patlu in the Game of Zones,Suhas Kadav,India,5/1/2021,2019,TV-Y7,87 min,"Children & Family Movies, Comedies, Music & Mu..."
8,s13,Movie,Je Suis Karl,Christian Schwochow,Germany,9/23/2021,2021,TV-MA,127 min,"Dramas, International Movies"
9,s940,Movie,Motu Patlu in Wonderland,Suhas Kadav,India,5/1/2021,2013,TV-Y7,76 min,"Children & Family Movies, Music & Musicals"


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Handle missing values (NaN) in the 'listed_in' column
# If a movie has no genre listed, we replace it with an empty string to avoid errors.
df['listed_in'] = df['listed_in'].fillna('')

# 2. Initialize the TF-IDF Vectorizer
# stop_words='english' tells it to ignore words like 'the', 'and', 'a'
tfidf = TfidfVectorizer(stop_words='english')

# 3. Fit and Transform the text data into a matrix of numbers
tfidf_matrix = tfidf.fit_transform(df['listed_in'])

# Let's see the shape (rows, columns) of our new matrix
print("Shape of TF-IDF Matrix:", tfidf_matrix.shape)

Shape of TF-IDF Matrix: (8790, 44)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between all movies
# This will create a massive square matrix (8790 x 8790)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Let's see the shape of this similarity matrix
print("Shape of Cosine Similarity Matrix:", cosine_sim.shape)

# Let's look at the similarity score between the first movie (index 0) and the second movie (index 1)
print(f"Similarity between movie 0 and 1: {cosine_sim[0, 1]:.4f}")

Shape of Cosine Similarity Matrix: (8790, 8790)
Similarity between movie 0 and 1: 0.0000


In [4]:
def get_recommendations(title, cosine_sim=cosine_sim):
    # 1. Get the index of the movie that matches the title
    # We find the row where the 'title' column matches our input title
    idx = df[df['title'] == title].index[0]
    
    # 2. Get the pairwise similarity scores of all movies with that movie
    # We enumerate this so we keep track of the index (i) and the score
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # 3. Sort the movies based on the similarity scores in descending order
    # key=lambda x: x[1] means "sort based on the second item in the tuple (the score)"
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # 4. Get the scores of the 10 most similar movies
    # We skip the first one [1:11] because the first one is always the movie itself (score 1.0)
    sim_scores = sim_scores[1:11]
    
    # 5. Get the movie indices
    movie_indices = [i[0] for i in sim_scores]
    
    # 6. Return the top 10 most similar movies' titles
    return df['title'].iloc[movie_indices]

# Let's test it out! 
# Pick any movie from your dataset. Let's try a very popular one:
print("Recommendations for 'Sankofa':")
print(get_recommendations('Sankofa'))

Recommendations for 'Sankofa':
54                  Ankahi Kahaniya
280       LSD: Love, Sex Aur Dhokha
382            La diosa del asfalto
509                 Eyes of a Thief
510          Pomegranates and Myrrh
559                 African America
684    I missed you: Director's Cut
775                   Into the Wind
805                        Fan Girl
841                   Tragic Jungle
Name: title, dtype: object


In [5]:
def get_smart_recommendations(title, cosine_sim=cosine_sim):
    # 1. Convert input to lowercase and check if it exists
    title = title.lower()
    
    # Create a lowercase version of our dataframe titles for matching
    df['title_lower'] = df['title'].str.lower()
    
    # Check if the movie exists
    if title not in df['title_lower'].values:
        return "Movie not found in the dataset. Please check the spelling."
    
    # 2. Get the index of the first occurrence of the movie
    idx = df[df['title_lower'] == title].index[0]
    
    # 3. Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # 4. Sort based on scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # 5. Get top 10 (skipping the movie itself)
    sim_scores = sim_scores[1:11]
    
    # 6. Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    
    # 7. Return the titles
    return df['title'].iloc[movie_indices]

# Test with a lowercase input to prove it works
print("Recommendations for 'sankofa' (lowercase):")
print(get_smart_recommendations('sankofa'))

print("\nRecommendations for a movie that doesn't exist:")
print(get_smart_recommendations('Avatar 3'))

Recommendations for 'sankofa' (lowercase):
54                  Ankahi Kahaniya
280       LSD: Love, Sex Aur Dhokha
382            La diosa del asfalto
509                 Eyes of a Thief
510          Pomegranates and Myrrh
559                 African America
684    I missed you: Director's Cut
775                   Into the Wind
805                        Fan Girl
841                   Tragic Jungle
Name: title, dtype: object

Recommendations for a movie that doesn't exist:
Movie not found in the dataset. Please check the spelling.


In [6]:
import pandas as pd
import numpy as np
import re

# 1. Load Data
df = pd.read_csv('Dataset.csv')

# --- MASTER PREPROCESSING STEP ---

# Edge Case 1 & 2: Handle Missing Values & Convert text to lowercase & strip whitespace
df['title_clean'] = df['title'].fillna('').astype(str).str.lower().str.strip()
df['listed_in'] = df['listed_in'].fillna('')

# Edge Case 3: Clean the genres (remove weird spacing, normalize commas)
# This replaces multiple spaces with a single space, and ensures commas are followed by a space
def clean_text(text):
    # Replace non-alphanumeric characters (except commas and spaces) with space
    text = re.sub(r'[^a-zA-Z0-9, ]', '', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['listed_in_clean'] = df['listed_in'].apply(clean_text)

# Edge Case 4: Drop rows where BOTH title and genres are empty (useless data)
df = df[(df['title_clean'] != '') & (df['listed_in_clean'] != '')].reset_index(drop=True)

print(f"Data cleaned! Total rows remaining: {len(df)}")
print(df[['title', 'listed_in']].head(3))

Data cleaned! Total rows remaining: 8790
                  title                                          listed_in
0  Dick Johnson Is Dead                                      Documentaries
1             Ganglands  Crime TV Shows, International TV Shows, TV Act...
2         Midnight Mass                 TV Dramas, TV Horror, TV Mysteries


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Vectorize the CLEANED genres
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['listed_in_clean'])

# Compute Cosine Similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Matrix shape: {cosine_sim.shape}")

Matrix shape: (8790, 8790)


In [8]:
def get_ultimate_recommendations(title, df=df, cosine_sim=cosine_sim, top_n=10):
    # 1. Standardize input: lowercase and strip spaces
    search_title = str(title).lower().strip()
    
    # 2. Find ALL matching indices (handles Duplicate Titles)
    # This returns a list of indices where the cleaned title matches
    matching_indices = df[df['title_clean'] == search_title].index.tolist()
    
    # Edge Case: Movie not found
    if not matching_indices:
        return f"❌ Error: '{title}' not found in the dataset. Please check spelling."
    
    # 3. If there are duplicates (e.g., two movies with the same name), we pick the first one.
    # In a real system, we would ask the user to clarify (e.g., "Did you mean the 1995 or 2018 version?")
    idx = matching_indices[0]
    
    # 4. Get pairwise similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # 5. Sort by score (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # 6. Get top N (skip the first one because it's the movie itself)
    sim_scores = sim_scores[1:top_n+1]
    
    # 7. Get the movie indices
    movie_indices = [i[0] for i in sim_scores]
    
    # 8. Return a clean DataFrame with Title, Genres, and Similarity Score
    # This is much more professional than just returning a list of titles
    results = df[['title', 'listed_in']].iloc[movie_indices].copy()
    results['similarity_score'] = [round(score, 4) for index, score in sim_scores]
    
    return results

# --- TESTING EDGE CASES ---

print("Test 1: Normal Input ('Sankofa')")
print(get_ultimate_recommendations('Sankofa').head(3))
print("\n" + "="*50 + "\n")

print("Test 2: Messy Input ('  sAnKoFa  ')")
print(get_ultimate_recommendations('  sAnKoFa  ').head(3))
print("\n" + "="*50 + "\n")

print("Test 3: Non-existent Input ('Harry Potter and the Cursed Child Movie')")
print(get_ultimate_recommendations('Harry Potter and the Cursed Child Movie'))

Test 1: Normal Input ('Sankofa')
                         title  \
54             Ankahi Kahaniya   
280  LSD: Love, Sex Aur Dhokha   
382       La diosa del asfalto   

                                            listed_in  similarity_score  
54   Dramas, Independent Movies, International Movies               1.0  
280  Dramas, Independent Movies, International Movies               1.0  
382  Dramas, Independent Movies, International Movies               1.0  


Test 2: Messy Input ('  sAnKoFa  ')
                         title  \
54             Ankahi Kahaniya   
280  LSD: Love, Sex Aur Dhokha   
382       La diosa del asfalto   

                                            listed_in  similarity_score  
54   Dramas, Independent Movies, International Movies               1.0  
280  Dramas, Independent Movies, International Movies               1.0  
382  Dramas, Independent Movies, International Movies               1.0  


Test 3: Non-existent Input ('Harry Potter and the Cursed Chil